# Kaggle Submission — Stanford RNA 3D Folding

Protenix (AF3) + LoRA fine-tuning + template search + diversity ensemble.

**Runtime constraints**: 29GB RAM, 20GB disk, 16GB VRAM, 12h, no internet.

In [ ]:
# === Phase 0: Install dependencies from offline wheels ===
import sys
import os
import time

TOTAL_START = time.time()

# Install from pre-uploaded Kaggle dataset (offline)
DEPS_DIR = "/kaggle/input/protenix-rna-deps"
if os.path.exists(DEPS_DIR):
    os.system(f"{sys.executable} -m pip install --no-index --find-links {DEPS_DIR} "
              f"protenix biopython einops peft -q")
else:
    print("WARNING: offline deps not found, trying online install")
    os.system(f"{sys.executable} -m pip install protenix biopython einops peft -q")

# Add code to path
sys.path.insert(0, '/kaggle/input/3drna-cc-code')  # uploaded as Kaggle dataset
# Or if code is in the notebook itself, use local path
if os.path.exists('/kaggle/working/3drna_cc'):
    sys.path.insert(0, '/kaggle/working/3drna_cc')

print(f"Setup time: {time.time()-TOTAL_START:.1f}s")

In [ ]:
# === Phase 1: Imports & Configuration ===
import numpy as np
import pandas as pd
from pathlib import Path

# Paths (Kaggle environment)
DATA_DIR = Path("/kaggle/input/stanford-rna-3d-folding-2")
MODEL_DIR = Path("/kaggle/input/protenix-rna-weights")
LORA_DIR = MODEL_DIR / "lora_weights"
OUTPUT_DIR = Path("/kaggle/working")
MSA_DIR = DATA_DIR / "MSA"
PDB_RNA_DIR = DATA_DIR / "PDB_RNA"

print(f"Data dir exists: {DATA_DIR.exists()}")
print(f"Model dir exists: {MODEL_DIR.exists()}")
print(f"MSA dir exists: {MSA_DIR.exists()}")

In [ ]:
# === Phase 2: Load Test Data ===
test_seq = pd.read_csv(DATA_DIR / "test_sequences.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Fill NaN string columns
for col in ['stoichiometry', 'all_sequences', 'ligand_ids', 'ligand_smiles']:
    if col in test_seq.columns:
        test_seq[col] = test_seq[col].fillna('')

print(f"Test targets: {len(test_seq)}")
print(f"Submission rows: {len(sample_sub)}")
print(f"Test target IDs: {list(test_seq['target_id'])}")
print(f"\nSequence lengths:")
for _, row in test_seq.iterrows():
    print(f"  {row['target_id']}: {len(row['sequence'])} nt")

In [ ]:
# === Phase 3: Initialize Model ===
from src.model.protenix_runner import ProtenixRunner

runner = ProtenixRunner(
    model_dir=MODEL_DIR,
    lora_dir=LORA_DIR if LORA_DIR.exists() else None,
    device='cuda',
)
print("Model initialized")

In [ ]:
# === Phase 4: Template Search + Feature Building + Inference ===
from src.data.featurizer import build_protenix_input
from src.data.loader import load_msa, parse_stoichiometry
from src.template.template_search import search_templates
from src.postprocess.clash_fix import fix_all
from src.ensemble.diversity_selector import select_best_five

all_predictions = {}  # target_id -> list of 5 (L,3) arrays

for idx, row in test_seq.iterrows():
    tid = row['target_id']
    t0 = time.time()
    print(f"\n{'='*60}")
    print(f"Target {idx+1}/{len(test_seq)}: {tid} ({len(row['sequence'])} nt)")
    
    # --- Template search ---
    try:
        templates = search_templates(
            tid, row['sequence'],
            pdb_seqres_path=PDB_RNA_DIR / 'pdb_seqres_NA.fasta',
            pdb_dates_path=PDB_RNA_DIR / 'pdb_release_dates_NA.csv',
            pdb_dir=PDB_RNA_DIR,
        )
        print(f"  Templates: {len(templates)}")
    except Exception as e:
        print(f"  Template search failed: {e}")
        templates = []
    
    # --- Generate diverse predictions ---
    all_preds = []
    
    # Multiple seed groups for diversity
    seed_groups = [
        list(range(1, 4)),    # seeds 1-3
        list(range(4, 7)),    # seeds 4-6
        list(range(7, 10)),   # seeds 7-9
    ]
    
    for sg_idx, seeds in enumerate(seed_groups):
        inp = build_protenix_input(
            target_id=tid,
            sequence=row['sequence'],
            stoichiometry=row.get('stoichiometry', ''),
            all_sequences=row.get('all_sequences', ''),
            ligand_ids=row.get('ligand_ids', ''),
            ligand_smiles=row.get('ligand_smiles', ''),
            msa_dir=MSA_DIR,
            msa_seed=sg_idx * 100 if sg_idx > 0 else None,  # different MSA subsamples
            template_hits=templates,
            model_seeds=seeds,
        )
        
        try:
            cifs = runner.predict_from_dict(inp)
            for cif in cifs:
                coords = runner.extract_c1_prime(cif)
                all_preds.append({
                    'coords': coords,
                    'source': f'group_{sg_idx}',
                })
        except Exception as e:
            print(f"  WARNING: seed group {sg_idx} failed: {e}")
    
    print(f"  Raw predictions: {len(all_preds)}")
    
    # --- Post-process ---
    for p in all_preds:
        p['coords'] = fix_all(p['coords'])
    
    # --- Select best 5 ---
    if all_preds:
        selected = select_best_five(all_preds, strategy='maxmin_diversity', n_select=5)
    else:
        # Fallback: zeros
        seq_len = len(row['sequence'])
        selected = [np.zeros((seq_len, 3))] * 5
    
    # Pad to exactly 5
    while len(selected) < 5:
        selected.append(selected[0].copy())
    
    all_predictions[tid] = selected[:5]
    
    elapsed = time.time() - t0
    total_elapsed = time.time() - TOTAL_START
    print(f"  Time: {elapsed:.1f}s | Total: {total_elapsed/60:.1f}min")

print(f"\n{'='*60}")
print(f"All targets predicted. Total time: {(time.time()-TOTAL_START)/60:.1f} min")

In [ ]:
# === Phase 5: Build Submission ===
from src.submission.formatter import format_submission, validate_submission

submission = format_submission(
    all_predictions,
    sample_submission_path=DATA_DIR / 'sample_submission.csv',
    output_path=OUTPUT_DIR / 'submission.csv',
)

is_valid = validate_submission(
    submission,
    sample_submission_path=DATA_DIR / 'sample_submission.csv',
)

total_time = time.time() - TOTAL_START
print(f"\nTotal runtime: {total_time/60:.1f} min ({total_time/3600:.2f} hours)")
print(f"Submission shape: {submission.shape}")
submission.head()